### Scenario: A knowledge management company wants agents to retrieve organizational information.

### Tasks: Build RAG-enabled agents using embeddings, vector databases, semantic search,
### and enterprise documents.

In [ ]:
! pip install -U langchain langchain-openai langchain-community langchain-chroma langchain-huggingface chromadb faiss-cpu pypdf python-dotenv

In [ ]:
import os

from dotenv import load_dotenv

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
from langchain_openai import ChatOpenAI
import os

load_dotenv()

openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

if not openrouter_api_key:
    raise ValueError("OPENROUTER_API_KEY was not found in the environment or .env file.")

client = OpenAI(
    api_key=openrouter_api_key,
    base_url="https://openrouter.ai/api/v1"
)

llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    api_key=openrouter_api_key,
    base_url="https://openrouter.ai/api/v1"
)

print("OpenRouter client and chat model ready")

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = "financial_policy_guidelines_and_example.pdf"

loader = PyPDFLoader(pdf_path)

documents = loader.load()

print("PDF loaded successfully")
print("Number of pages:", len(documents))

In [ ]:
print(documents)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print("Documents split successfully")
print("Total chunks:", len(chunks))

In [ ]:
print(chunks[0].page_content)

In [ ]:
embedding = client.embeddings.create(
    model="liquid/lfm-2.5-embedding-350m:free",
    input=[chunk.page_content for chunk in chunks]
)

print("Embeddings generated successfully")
print("Total embeddings:", len(embedding.data))
print("Embedding dimension:", len(embedding.data[0].embedding))

In [ ]:
from langchain_core.embeddings import Embeddings

class OpenRouterEmbeddings(Embeddings):

    def embed_documents(self, texts):
        response = client.embeddings.create(
            model="liquid/lfm-2.5-embedding-350m:free",
            input=texts
        )
        return [item.embedding for item in response.data]

    def embed_query(self, text):
        response = client.embeddings.create(
            model="liquid/lfm-2.5-embedding-350m:free",
            input=text
        )
        return response.data[0].embedding


embeddings = OpenRouterEmbeddings()

print("OpenRouter embeddings ready")

In [ ]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

print("FAISS vector store created successfully")

In [ ]:
vectorstore.save_local("faiss_index")

print("FAISS index saved successfully")

In [ ]:
# Create a retriever using similarity search

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print("Similarity search retriever created successfully")

In [ ]:
query = input("Enter your query: ")

retrieved_docs = retriever.invoke(query)

context = "\n\n".join(
    doc.page_content
    for doc in retrieved_docs
)

prompt = f"""
Answer the question using only the provided context.

Context:
{context}

Question:
{query}

If the answer is not present in the context, say:
"I could not find this information in the provided documents."
"""

response = llm.invoke(prompt)



In [ ]:
answer=response.content

In [ ]:
type (answer)

In [ ]:
print(answer)